In [4]:
print("success")

success


In [5]:
!pip install torch==2.6.0 transformers==4.46.3 tokenizers==0.20.3 einops addict easydict --quiet

In [6]:
!pip install flash-attn==2.7.3 --no-build-isolation --quiet

In [7]:
!pip install pdf2image pillow --quiet
!apt-get install -y poppler-utils > /dev/null

In [8]:
from transformers import AutoModel, AutoTokenizer
import torch
import os
from pdf2image import convert_from_path
from PIL import Image
from pathlib import Path
import tempfile
import sys
from io import StringIO
import re
os.environ["CUDA_VISIBLE_DEVICES"] = '0'
model_name = 'deepseek-ai/DeepSeek-OCR'

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, use_safetensors=True)
model = model.eval().cuda().to(torch.bfloat16)

2026-05-14 08:56:39.774336: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778748999.929676     284 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778748999.973624     284 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at deepseek-ai/DeepSeek-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
def extract_markdown_from_output(output_text):
    """Extract clean markdown from the model's output text"""
    lines = output_text.split('\n')
    markdown_lines = []
    
    for line in lines:
        # Skip lines with special tokens and detection coordinates
        if '<|ref|>' in line or '<|det|>' in line or '<|/ref|>' in line or '<|/det|>' in line:
            # Extract the actual text between tags
            # Pattern: <|ref|>type<|/ref|><|det|>coords<|/det|>\nActual text
            continue
        elif line.startswith('==') or 'image size:' in line or 'tokens:' in line or 'compression ratio:' in line:
            continue
        elif line.strip():
            markdown_lines.append(line)
    
    return '\n'.join(markdown_lines)

In [11]:
def capture_model_output(func, *args, **kwargs):
    """Capture stdout from model.infer() call"""
    old_stdout = sys.stdout
    sys.stdout = captured_output = StringIO()
    
    try:
        result = func(*args, **kwargs)
        output = captured_output.getvalue()
    finally:
        sys.stdout = old_stdout
    
    return result, output

In [12]:
def ocr_pdf_to_markdown(pdf_path, output_md_path, dpi=200, base_size=1024, image_size=640):
    """
    Convert a scanned PDF to markdown using DeepSeek-OCR
    
    Args:
        pdf_path: Path to the input PDF file
        output_md_path: Path to save the output markdown file
        dpi: DPI for PDF to image conversion (higher = better quality, slower)
        base_size: Base size parameter for OCR model
        image_size: Image size parameter for OCR model
    """
    print(f"Processing PDF: {pdf_path}")
    
    # Convert PDF to images
    print("Converting PDF pages to images...")
    images = convert_from_path(pdf_path, dpi=dpi)
    print(f"Found {len(images)} pages")
    
    # Create temporary directory for intermediate images
    with tempfile.TemporaryDirectory() as temp_dir:
        all_markdown = []
        
        for page_num, image in enumerate(images, 1):
            print(f"\nProcessing page {page_num}/{len(images)}...")
            
            # Save temporary image
            temp_image_path = os.path.join(temp_dir, f"page_{page_num}.png")
            image.save(temp_image_path, 'PNG')
            
            # OCR the image and capture output
            prompt = "<image>\n<|grounding|>Convert the document to markdown."
            
            try:
                # Capture the printed output from model.infer()
                res, output = capture_model_output(
                    model.infer,
                    tokenizer,
                    prompt=prompt,
                    image_file=temp_image_path,
                    output_path=temp_dir,
                    base_size=base_size,
                    image_size=image_size,
                    crop_mode=True,
                    save_results=False,
                    test_compress=True
                )
                
                # Extract markdown from the captured output
                markdown_text = extract_markdown_from_output(output)
                
                if markdown_text.strip():
                    all_markdown.append(f"# Page {page_num}\n\n{markdown_text}\n\n---\n")
                    print(f"✓ Successfully extracted {len(markdown_text)} characters from page {page_num}")
                else:
                    all_markdown.append(f"# Page {page_num}\n\n[No text extracted from this page]\n\n---\n")
                    print(f"⚠ No text extracted from page {page_num}")
                
            except Exception as e:
                print(f"✗ Error processing page {page_num}: {e}")
                all_markdown.append(f"# Page {page_num}\n\n[Error processing this page: {e}]\n\n---\n")
    
    # Save combined markdown
    print(f"\nSaving markdown to: {output_md_path}")
    with open(output_md_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(all_markdown))
    
    print("✓ Done!")
    return output_md_path

In [13]:
def batch_process_pdfs(pdf_dir, output_dir, dpi=200):
    """
    Process multiple PDFs in a directory
    
    Args:
        pdf_dir: Directory containing PDF files
        output_dir: Directory to save markdown files
        dpi: DPI for PDF to image conversion
    """
    os.makedirs(output_dir, exist_ok=True)
    
    pdf_files = list(Path(pdf_dir).glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")
    
    for pdf_file in pdf_files:
        output_md = os.path.join(output_dir, f"{pdf_file.stem}.md")
        ocr_pdf_to_markdown(str(pdf_file), output_md, dpi=dpi)

In [16]:
# Example usage for single PDF
pdf_path = '/kaggle/input/datasets/abulkashemjunaid/short-pdf/hahahah_3-8.pdf'
output_md_path = '/kaggle/working/shortpdf.md'

In [17]:
ocr_pdf_to_markdown(pdf_path, output_md_path, dpi=200)

Processing PDF: /kaggle/input/datasets/abulkashemjunaid/short-pdf/hahahah_3-8.pdf
Converting PDF pages to images...
Found 6 pages

Processing page 1/6...


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Cal

✓ Successfully extracted 3618 characters from page 1

Processing page 2/6...


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


✓ Successfully extracted 3955 characters from page 2

Processing page 3/6...


...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


✓ Successfully extracted 3588 characters from page 5

Processing page 6/6...


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


✓ Successfully extracted 3280 characters from page 6

Saving markdown to: /kaggle/working/shortpdf.md
✓ Done!


'/kaggle/working/shortpdf.md'

In [18]:
# Example usage for single PDF
pdf_path = '/kaggle/input/datasets/abulkashemjunaid/legal-dataset007/image.pdf'
output_md_path = '/kaggle/working/image.md'

In [19]:
ocr_pdf_to_markdown(pdf_path, output_md_path, dpi=200)

Processing PDF: /kaggle/input/datasets/abulkashemjunaid/legal-dataset007/image.pdf
Converting PDF pages to images...
Found 2 pages

Processing page 1/2...


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


✓ Successfully extracted 2604 characters from page 1

Processing page 2/2...


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


✓ Successfully extracted 1914 characters from page 2

Saving markdown to: /kaggle/working/image.md
✓ Done!


'/kaggle/working/image.md'

In [20]:
# Example usage for single PDF
pdf_path = '/kaggle/input/datasets/abulkashemjunaid/handwrittendataset/fa7f7a362c7843d6b18aa7c990fbae2e.pdf'
output_md_path = '/kaggle/working/handwritten.md'

In [21]:
ocr_pdf_to_markdown(pdf_path, output_md_path, dpi=200)

Processing PDF: /kaggle/input/datasets/abulkashemjunaid/handwrittendataset/fa7f7a362c7843d6b18aa7c990fbae2e.pdf
Converting PDF pages to images...
Found 1 pages

Processing page 1/1...


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


✓ Successfully extracted 739 characters from page 1

Saving markdown to: /kaggle/working/handwritten.md
✓ Done!


'/kaggle/working/handwritten.md'